In [2]:
!pip install pandas
!pip install yfinance
!pip install hmmlearn
!pip install plotly
!pip install numpy
!pip install matplotlib

In [3]:
import numpy
from hmmlearn import hmm
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

#regimes (trending, mean reverting, high volatility, low volatility)

In [4]:
import yfinance as yf

def clean_yfinance_data(ticker, start, end):
    """Baixa dados e remove MultiIndex se existir"""
    data = yf.download(ticker, start=start, end=end)

    # Se tiver MultiIndex nas colunas, remove
    if isinstance(data.columns, pd.MultiIndex):
        data.columns = data.columns.droplevel(1)

    return data['Close']

# Usar a função
sp500 = clean_yfinance_data("^GSPC", "2000-01-01", "2025-12-31")
ibov = clean_yfinance_data("^BVSP", "2000-01-01", "2025-12-31")

/tmp/ipykernel_4245/1861812893.py:5: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, start=start, end=end)
[*********************100%***********************]  1 of 1 completed
/tmp/ipykernel_4245/1861812893.py:5: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, start=start, end=end)
[*********************100%***********************]  1 of 1 completed


In [9]:
def prepare_features(prices, window=90):
    """prepara feature para os 4 estados"""
    feature = pd.DataFrame(prices)

    feature['returns'] = np.log(prices / prices.shift(1))

    feature['volatility'] = feature['returns'].rolling(window).std()

    feature['momentum'] = (prices / prices.rolling(window).mean()) - 1

    feature['mean_reversion'] = -feature['returns'].rolling(window).mean()

    feature['vol_regime'] = feature['volatility'] / feature['volatility'].rolling(252).mean()

    return feature.dropna()


In [12]:
df = pd.read_csv('markets.csv', parse_dates=True)
df.set_index('Date', inplace=True)
df



,ibovespa_br,s&p500_eua,dax_alemanha,shanghai_china,nikkei_japao,merv_argentina,ftse_uk
Date,,,,,,,
2000-01-03,16930.0,1455.219971,6750.759766,NaN,NaN,552.0,NaN
2000-01-04,15851.0,1399.420044,6586.950195,1406.370972,19002.859375,523.0,6665.899902
2000-01-05,16245.0,1402.109985,6502.069824,1409.682007,18542.550781,533.0,6535.899902
2000-01-06,16107.0,1403.449951,6474.919922,1463.942017,18168.269531,529.0,6447.200195
2000-01-07,16309.0,1441.469971,6780.959961,1516.604004,18193.410156,522.0,6504.799805
...,...,...,...,...,...,...,...
2026-03-24,182509.0,6556.370117,22636.910156,3881.280029,52252.281250,NaN,9965.200195
2026-03-25,185424.0,6591.899902,22957.080078,3931.836914,53749.621094,2805317.0,10106.799805
2026-03-26,182733.0,6477.160156,22612.970703,3889.083008,53603.648438,2769369.0,9972.200195


In [13]:
from sklearn.preprocessing import StandardScaler
from hmmlearn import hmm


def identifying_regimes(features_df, n_states=4):

    X = features_df.values

    scaler = StandardScaler()

    X_scaled = scaler.fit_transform(X)

    model = hmm.GaussianHMM(
        n_components=n_states,
        covariance_type='full',
        n_iter=1000,
        random_state=42
    )

    model.fit(X_scaled)

    regimes = model.predict(X_scaled)

    features_df['regime'] = regimes

    return features_df

In [16]:
regimes_markets = {}

for market in df.columns:

    try:

        prices = df[market].dropna()

        features = prepare_features(prices)

        regimes = identifying_regimes(features)

        regimes_markets[market] = regimes

        print(f'{market} OK')

    except Exception as e:

        print(f'Erro em {market}: {e}')

regimes

ibovespa_br OK
s&p500_eua OK
dax_alemanha OK
shanghai_china OK
nikkei_japao OK
merv_argentina OK
ftse_uk OK


,ftse_uk,returns,volatility,momentum,mean_reversion,vol_regime,regime
Date,,,,,,,
2001-05-10,5964.000000,0.011857,0.012934,0.003060,0.000471,1.144672,2
2001-05-11,5896.799805,-0.011332,0.012963,-0.007727,0.000512,1.147680,2
2001-05-14,5690.500000,-0.035612,0.013290,-0.041815,0.000662,1.176990,2
2001-05-15,5842.899902,0.026429,0.013347,-0.015523,0.000633,1.182274,2
2001-05-16,5884.000000,0.007010,0.013368,-0.008014,0.000578,1.184506,2
...,...,...,...,...,...,...,...
2026-03-24,9965.200195,0.007150,0.007605,-0.013847,-0.000177,1.058191,3
2026-03-25,10106.799805,0.014109,0.007647,-0.000283,-0.000458,1.063026,3
2026-03-26,9972.200195,-0.013407,0.007780,-0.013919,-0.000336,1.080390,3


,ftse_uk
Date,
2000-01-04,6665.899902
2000-01-05,6535.899902
2000-01-06,6447.200195
2000-01-07,6504.799805
2000-01-10,6607.700195
...,...
2026-03-24,9965.200195
2026-03-25,10106.799805
2026-03-26,9972.200195


**Bellman Equation and HMMs models***

Belmann Equation used for:  making decisions and control (maximize a reward).

HMM used for pattern recognition and inference (guessing hidden sitaution based in clues)


The solution it will be implemented to 2 steps. The first steps consist in training for the regimes with macroeconomic data. Second step consist in the with this signals, how i can build and redefine weights for my assets.

1. Making decodes process (e.g 75% stagflation, 25% growth).

2. Using bellman equation to made the best policy for the next regimes. Using data like price. value compared the dy.



In [17]:
def belmann_equation(states, actions, transactions, rewards, discount_factor=0.99, theta=1e-5):
    """
      V(s): The valeu state sss, which represents the long-term reward of being state.
      R(s,a): The immediate reward received for taking action aaa in state sss
      Y: factor discount (between 0 and 1) that determines importance of future rewards compared to immediate rewards.
      P(s'|s,a): The probability  of transaction state s' from state sss by taking action a.
      discount_factor: float (gamma) valuing futures states relative to current ones.
      theta: float, accuracy threshold for convergence of the Bellman equation. (WHEN ALGORITHM STOPS)
      max_a: The optimal action that maximizes the expected value future rewards.

      V(s): the regime
      R(s,a): rebalacing for the stocks, signals to more options in determinates sectors.
      Y: We can use risk return to calculate this factors.
      P(s' | s,a): we can use the matrix transactions for the policies
    """

    v = {s: 0.0 for s in states}
    policy = {s: None for s in states}

    #algorithm iteration

    while True:
      delta = 0
      for s in states:
        v_old = v[s]
        max_q_value = float('-inf')
        best_action = None

        for a in actions:
          q_value = rewards.get((s, a), 0)
          expected_future = 0

        for next_states in states:
          prob = transactions.get((s, a, next_states), 0)
          expected_future += prob * (discount_factor * v[next_states]) #projection inference for the prob future value

        q_value += expected_future

        if q_value > max_q_value:
          max_q_value = q_value
          best_action = a

        v[s] = max_q_value
        policy[s] = best_action
        delta = max(delta, abs(v_old - v[s])) #verify if have lost value between to groups

      if delta < theta:
        break

    return v, policy
